In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u
import logging


In [2]:

def setup_logging(verbose=False):
    """
    Call this once at the top of the notebook.
    If verbose=True, DEBUG messages will show; otherwise only INFO+.
    """
    name = "Coma_GCs_debug"
    
    # Remove any root handlers (if I had root-level logging before)
    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)

    # Get named logger and clear *its* handlers + disable propagation
    logger = logging.getLogger(name)
    for h in logger.handlers[:]:
        logger.removeHandler(h)
    logger.propagate = False

    # Set logger’s level
    logger.setLevel(logging.DEBUG)   # we capture everything here

    # INFO-only handler (no timestamp)
    info_handler = logging.StreamHandler()
    info_handler.setLevel(logging.INFO)
    info_handler.addFilter(lambda rec: rec.levelno == logging.INFO)
    info_handler.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(info_handler)

    # non-INFO handler (timestamped, DEBUG or WARNING+)
    other = logging.StreamHandler()
    other.setLevel(logging.DEBUG if verbose else logging.WARNING)
    other.addFilter(lambda rec: rec.levelno != logging.INFO)
    # allow only your logger’s DEBUG (if you still want a name-filter):
    other.addFilter(lambda rec: rec.levelno != logging.DEBUG or rec.name == name)
    fmt = "[%(levelname)s] %(asctime)s.%(msecs)03d %(message)s"
    other.setFormatter(logging.Formatter(fmt, datefmt="%H:%M:%S"))
    logger.addHandler(other)


In [3]:

# Set verbosity here:
setup_logging(verbose=True)   # or False

# testing logging
logger = logging.getLogger("Coma_GCs_debug")
logger.debug("Debug messages are ON")
logger.info("Info messages are always shown")
logger.warning("Warnings also show up")


[DEBUG] 11:03:06.220 Debug messages are ON
Info messages are always shown
[WARNING] 11:03:06.223 Warnings also show up


## Attempt to obtain full galaxy dataset from archives

Although a manual dataset has been determined from the starting point of the 'notable' galaxies in the Madrid 2018 paper, we have criteria now which should allow the extraction of galaxies from the Simbad DB, specifically, we can use the redshift range of objects which are classified as galaxies within the target area. A further filter by V magnitude will also be applied.


In [4]:
from astroquery.simbad import Simbad

adql = """
SELECT
  b.main_id,
  b.ra, b.dec,
  b.otype,
  b.rvz_redshift AS z,
  f."V" AS Vmag
FROM basic AS b
JOIN allfluxes AS f ON b.oid = f.oidref
WHERE b.ra  BETWEEN 194.76933 AND 195.28932
  AND b.dec BETWEEN  27.82581 AND  28.12152
  AND b.otype = 'G..'                      -- galaxies & subtypes
  AND b.rvz_redshift BETWEEN 0.01241 AND 0.044067
  AND f."V" < 20
ORDER BY Vmag ASC
"""

tbl = Simbad.query_tap(adql)         # Astropy Table
df = tbl.to_pandas()                 # -> pandas DataFrame
df


,main_id,ra,dec,otype,z,vmag
0,NGC 4889,195.033738,27.977025,EmG,0.021500,11.300000
1,NGC 4874,194.898789,27.959248,LIN,0.023910,12.710000
2,IC 4051,195.226929,28.007639,EmG,0.016620,13.500000
3,NGC 4869,194.847328,27.911592,rG,0.022879,13.520000
4,NGC 4908,195.214762,28.042874,LIN,0.029160,13.580000
...,...,...,...,...,...,...
203,GMP 2707,195.097850,28.050510,G,0.023603,19.889999
204,SDSS J130005.76+280212.1,195.024015,28.036713,G,0.019400,19.910000
205,SDSS J130016.67+275638.6,195.069472,27.944075,LSB,0.017899,19.930000
206,SDSS J125934.40+275942.8,194.893346,27.995247,G,0.017429,19.940001


In [5]:
# SIMBAD + NED with galaxy dimensions & V<20
from astroquery.simbad import Simbad
from astroquery.ipac.ned import Ned
import pandas as pd
import numpy as np

# --- 1) Pull the base sample from SIMBAD TAP (V-band via allfluxes, z via rvz_redshift)
adql = """
SELECT
  b.main_id,
  b.ra, b.dec,                  -- degrees (ICRS)
  b.otype,
  b.rvz_redshift AS z,
  f."V" AS Vmag                 -- V magnitude
FROM basic AS b
JOIN allfluxes AS f ON b.oid = f.oidref
WHERE b.ra  BETWEEN 194.76933 AND 195.28932
  AND b.dec BETWEEN  27.82581 AND  28.12152
  AND b.otype = 'G..'                      -- galaxies & subtypes
  AND b.rvz_redshift BETWEEN 0.01241 AND 0.044067
  AND f."V" < 19
"""

base = Simbad.query_tap(adql)          # Astropy Table
df = base.to_pandas()

# --- 2) Batch-fetch galaxy dimensions from SIMBAD (major/minor/PA).
#       These arrive as arcminutes (at B=25 isophote), column names: galdim_majaxis, ...
s = Simbad()
s.add_votable_fields("dim")            # -> galdim_majaxis, galdim_minaxis, galdim_angle, ...
dims = s.query_objects(df["main_id"].tolist()).to_pandas()

# Keep only the columns we need and make sure they’re unique by main_id
keep = ["main_id", "Vmag", "galdim_majaxis", "galdim_minaxis", "galdim_angle", "galdim_qual", "galdim_bibcode"]
dims = dims[[c for c in keep if c in dims.columns]].drop_duplicates(subset="main_id", keep="first")

# Merge onto the base sample
out = df.merge(dims, on="main_id", how="left")

# Convert SIMBAD major axis (arcmin) -> arcsec and tag the source
out["major_axis_arcsec"] = out.get("galdim_majaxis") * 60.0
out["dim_source"] = np.where(out["major_axis_arcsec"].notna(), "SIMBAD", None)

# --- 3) Fallback to NED for rows still missing a major axis
def ned_major_axis_arcsec(name: str):
    """Return a best-effort major-axis in arcsec from NED 'diameters' table (or np.nan)."""
    try:
        t = Ned.get_table(name, table='diameters')  # may raise or return empty
    except Exception:
        return np.nan, None, None  # value, pa_deg, refcode

    if t is None or len(t) == 0:
        return np.nan, None, None

    # Try to find a column with 'major' (prefer one that mentions arcsec)
    maj_cols = [c for c in t.colnames if 'major' in c.lower()]
    if not maj_cols:
        return np.nan, None, None
    # Prefer a column explicitly labeled in arcsec if present
    cmaj = next((c for c in maj_cols if 'arcsec' in c.lower() or '(")' in c.lower()), maj_cols[0])

    # NED often has multiple rows (different bands/methods). Take the largest numeric value.
    vals = pd.to_numeric(pd.Series(t[cmaj]), errors='coerce')
    if vals.notna().any():
        idx = int(vals.idxmax())
        val = float(vals.iloc[vals.argmax()])
        # Try to also grab PA and a reference code if present
        pa_col = next((c for c in t.colnames if 'pa' in c.lower() and 'deg' in c.lower()), None)
        ref_col = next((c for c in t.colnames if 'ref' in c.lower()), None)
        pa = float(pd.to_numeric(pd.Series(t[pa_col]), errors='coerce').iloc[idx]) if pa_col else None
        refcode = str(t[ref_col][idx]) if ref_col else None
        return val, pa, refcode

    return np.nan, None, None

mask = out["major_axis_arcsec"].isna()
if mask.any():
    fill_vals = []
    fill_pa = []
    fill_ref = []
    for name in out.loc[mask, "main_id"]:
        v, pa, refcode = ned_major_axis_arcsec(name)
        fill_vals.append(v)
        fill_pa.append(pa)
        fill_ref.append(refcode)

    # Assign back to the corresponding rows
    idx_mask = out.index[mask]  # index positions we attempted to fill
    
    # Make sure columns exist
    if "dim_ref" not in out.columns:
        out["dim_ref"] = pd.NA
    if "galdim_angle" not in out.columns:
        out["galdim_angle"] = np.nan
    
    # Build Series aligned to the masked index
    s_major = pd.Series(fill_vals, index=idx_mask, dtype="float64")
    s_pa    = pd.Series(fill_pa,   index=idx_mask, dtype="float64")
    s_ref   = pd.Series(fill_ref,  index=idx_mask, dtype="object")
    
    # Assign back exactly to those rows (lengths now match)
    out["major_axis_arcsec"] = out["major_axis_arcsec"].astype("float64")
    out.loc[idx_mask, "major_axis_arcsec"] = s_major.to_numpy(dtype="float64")
    # out.loc[idx_mask, "major_axis_arcsec"] = s_major.values
    out.loc[idx_mask, "galdim_angle"]      = s_pa.values
    out.loc[idx_mask, "dim_ref"]           = s_ref.values
    
    # Mark source where we actually obtained a value from NED
    out.loc[idx_mask, "dim_source"] = np.where(
        s_major.notna(),
        "NED",
        out.loc[idx_mask, "dim_source"]  # keep existing (e.g., None) if still NaN
    )

# Tidy: order columns
cols = ["main_id", "ra", "dec", "otype", "z", "vmag",
        "major_axis_arcsec", "galdim_minaxis", "galdim_angle", "dim_source", "galdim_qual", "galdim_bibcode", "dim_ref"]
out = out[[c for c in cols if c in out.columns]]

out['MV'] = out['vmag'] - 35

# Show the DataFrame
out


,main_id,ra,dec,otype,z,vmag,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV
0,LEDA 3098454,195.075410,27.956592,GiC,0.021240,14.333000,23.714220,0.229277,143,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.667000
1,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,1.300000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-18.056000
2,LEDA 126763,195.055833,28.053417,GiC,0.027160,17.309999,23.923620,0.110128,81,SIMBAD,B,2023ApJS..269....3M,<NA>,-17.690001
3,GMP 3424,194.872189,27.942225,GiC,0.018723,18.773001,4.070000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.226999
4,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,7.150000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.740000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,IC 4021,195.061455,28.041300,LIN,0.019120,14.750000,25.321800,0.362693,49,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.250000
169,LEDA 126761,195.077274,28.097154,G,0.026080,17.040001,15.108181,0.185831,172,SIMBAD,B,2023ApJS..269....3M,<NA>,-17.959999
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.110000
171,IC 3973,194.878430,27.884206,AG?,0.015710,14.030000,49.606979,0.516492,137,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.970000


In [6]:
out['show_label'] = out["main_id"].astype("string").str.contains(
    r"(?i)\b(NGC|IC)\b", na=False
)


/var/folders/_d/2vszj0nd6532d6knr4c9lnrw0000gn/T/ipykernel_33688/914833855.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out['show_label'] = out["main_id"].astype("string").str.contains(


In [7]:
out.sort_values(by='MV', ascending=False)


,main_id,ra,dec,otype,z,vmag,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV,show_label
31,SDSS J130037.31+275441.0,195.155462,27.911388,G,0.020300,18.969999,NaN,NaN,<NA>,None,,,None,-16.030001,False
157,SDSS J130005.34+275628.8,195.022289,27.941354,G,0.023700,18.910000,NaN,NaN,<NA>,None,,,None,-16.090000,False
22,SDSS J130027.87+275916.2,195.116153,27.987850,G,0.032000,18.900000,NaN,NaN,<NA>,None,,,None,-16.100000,False
25,SDSS J130042.51+280325.4,195.177164,28.057068,AG?,0.019403,18.889999,4.380000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.110001,False
19,GMP 3308,194.904625,27.972333,G,0.025400,18.841999,1.630000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.158001,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,NGC 4908,195.214762,28.042874,LIN,0.029160,13.580000,57.341999,0.666983,57,SIMBAD,B,2023ApJS..269....3M,<NA>,-21.420000,True
117,NGC 4869,194.847328,27.911592,rG,0.022879,13.520000,49.683781,0.690853,47,SIMBAD,B,2023ApJS..269....3M,<NA>,-21.480000,True
100,IC 4051,195.226929,28.007639,EmG,0.016620,13.500000,79.127998,0.915775,103,SIMBAD,B,2023ApJS..269....3M,<NA>,-21.500000,True
33,NGC 4874,194.898789,27.959248,LIN,0.023910,12.710000,140.404205,2.260740,63,SIMBAD,B,2023ApJS..269....3M,<NA>,-22.290000,True


In [8]:
# collapse runs of whitespace to a single space + trim ends
out["main_id"] = (
    out["main_id"]
      .astype("string")                 # keeps NA values cleanly
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)
out["main_id"] = (
    out["main_id"].astype("string")
      .str.replace("\u00A0", " ", regex=False)  # NBSP -> normal space
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)
assert not out["main_id"].str.contains(r"\s{2,}", na=False).any()


In [9]:
# out_row = out.loc[out['main_id'] == 'NGC 4889']
out_row = out.iloc[23]
out_row

main_id                         NGC 4889
ra                            195.033738
dec                            27.977025
otype                                EmG
z                                 0.0215
vmag                                11.3
major_axis_arcsec             175.404602
galdim_minaxis                    1.8169
galdim_angle                          75
dim_source                        SIMBAD
galdim_qual                            B
galdim_bibcode       2023ApJS..269....3M
dim_ref                             <NA>
MV                                 -23.7
show_label                          True
Name: 23, dtype: object

In [10]:
out = out.rename(columns={
    "main_id": "name",
    "vmag": "mv",
}, errors="raise")
out

,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV,show_label
0,LEDA 3098454,195.075410,27.956592,GiC,0.021240,14.333000,23.714220,0.229277,143,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.667000,False
1,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,1.300000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-18.056000,False
2,LEDA 126763,195.055833,28.053417,GiC,0.027160,17.309999,23.923620,0.110128,81,SIMBAD,B,2023ApJS..269....3M,<NA>,-17.690001,False
3,GMP 3424,194.872189,27.942225,GiC,0.018723,18.773001,4.070000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.226999,False
4,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,7.150000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.740000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,IC 4021,195.061455,28.041300,LIN,0.019120,14.750000,25.321800,0.362693,49,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.250000,True
169,LEDA 126761,195.077274,28.097154,G,0.026080,17.040001,15.108181,0.185831,172,SIMBAD,B,2023ApJS..269....3M,<NA>,-17.959999,False
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.110000,False
171,IC 3973,194.878430,27.884206,AG?,0.015710,14.030000,49.606979,0.516492,137,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.970000,True


In [11]:
out['Re_arcsec'] = out['major_axis_arcsec'] / 8
out

,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV,show_label,Re_arcsec
0,LEDA 3098454,195.075410,27.956592,GiC,0.021240,14.333000,23.714220,0.229277,143,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.667000,False,2.964278
1,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,1.300000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-18.056000,False,0.162500
2,LEDA 126763,195.055833,28.053417,GiC,0.027160,17.309999,23.923620,0.110128,81,SIMBAD,B,2023ApJS..269....3M,<NA>,-17.690001,False,2.990453
3,GMP 3424,194.872189,27.942225,GiC,0.018723,18.773001,4.070000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.226999,False,0.508750
4,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,7.150000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.740000,False,0.893750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,IC 4021,195.061455,28.041300,LIN,0.019120,14.750000,25.321800,0.362693,49,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.250000,True,3.165225
169,LEDA 126761,195.077274,28.097154,G,0.026080,17.040001,15.108181,0.185831,172,SIMBAD,B,2023ApJS..269....3M,<NA>,-17.959999,False,1.888523
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.110000,False,5.332778
171,IC 3973,194.878430,27.884206,AG?,0.015710,14.030000,49.606979,0.516492,137,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.970000,True,6.200872


In [12]:
# manual row for cluster, for overall centering and plotting
manual = {
    "name": "Coma cluster",
    "ra": 195.01707133,               # center RA adjusted from BCG NGC 4889
    "dec": 27.977025,                 # center Dec too
    "otype": "CLUSTER",
    "z": 0.0215,                      # again, a representative z from NGC 4889
    # "Vmag": pd.NA,
    # "major_axis_arcsec": pd.NA,
    "dim_source": "MANUAL",
    "show_label": False
}

extra = pd.DataFrame([manual]).reindex(columns=out.columns, fill_value=pd.NA)
out = pd.concat([out, extra], ignore_index=True)


/var/folders/_d/2vszj0nd6532d6knr4c9lnrw0000gn/T/ipykernel_33688/2394890328.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([out, extra], ignore_index=True)


In [13]:
gals_df = out.copy()

In [14]:

# per-type counts
otype_counts = (
    gals_df["otype"].astype("string").str.strip()
      .value_counts(dropna=False)
      .rename_axis("otype")
      .reset_index(name="count")
)
otype_counts.head(10)


,otype,count
0,GiC,62
1,G,52
2,AG?,25
3,EmG,16
4,LIN,5
5,AGN,5
6,rG,3
7,GiG,3
8,LSB,2
9,CLUSTER,1


In [15]:
# minimal, practical cheat-sheet (not exhaustive)
otype_map = {
    # single galaxies
    "G": "Galaxy",
    "G?": "Galaxy candidate",
    "LSB": "Low-surface-brightness galaxy",
    "BCD": "Blue compact dwarf galaxy",
    "dG": "Dwarf galaxy",
    "IG": "Interacting galaxy",
    "EmG": "Emission-line galaxy",
    # activity/AGN subclasses
    "SyG": "Seyfert galaxy",
    "Sy1": "Seyfert 1",
    "Sy2": "Seyfert 2",
    "LIN": "LINER galaxy",
    "AGN": "Active galactic nucleus (galaxy)",
    "BLLac": "BL Lac object",
    "QSO": "Quasar",
    # membership in larger structures
    "GiP": "Galaxy in a pair",
    "GiG": "Galaxy in a group",
    "GiC": "Galaxy in a cluster",
    "BiC": "Brightest cluster galaxy (BCG)",
    # systems of galaxies
    "PaG": "Pair of galaxies",
    "GrG": "Group of galaxies",
    "CGG": "Compact group of galaxies",
    "ClG": "Cluster of galaxies",
    "SCG": "Supercluster of galaxies",
    # broad morphology (when present in otype)
    "E": "Elliptical galaxy",
    "S0": "Lenticular galaxy",
    "S": "Spiral galaxy",
    "SA": "Unbarred spiral",
    "SB": "Barred spiral",
    "SAB": "Weakly barred spiral",
    "Im": "Irregular galaxy",
    # alternates you might also see
    "GPair": "Galaxy pair",
    "GTrpl": "Galaxy triple",
    "GGroup": "Galaxy group",
}

lookup = (pd.DataFrame.from_dict(otype_map, orient="index", columns=["meaning"])
          .reset_index().rename(columns={"index": "otype"}))

otype_counts = otype_counts.merge(lookup, on="otype", how="left")
otype_counts.head(20)


,otype,count,meaning
0,GiC,62,Galaxy in a cluster
1,G,52,Galaxy
2,AG?,25,NaN
3,EmG,16,Emission-line galaxy
4,LIN,5,LINER galaxy
5,AGN,5,Active galactic nucleus (galaxy)
6,rG,3,NaN
7,GiG,3,Galaxy in a group
8,LSB,2,Low-surface-brightness galaxy
9,CLUSTER,1,NaN


In [16]:
family_bins = {
    "Active (AGN/Seyfert/QSO)": {"AGN","SyG","Sy1","Sy2","LIN","BLLac","QSO"},
    "Systems (pairs/groups/clusters)": {"PaG","GrG","CGG","ClG","SCG","GPair","GTrpl","GGroup"},
    "Members of systems": {"GiP","GiG","GiC","BiC"},
    "LSB/BCD/Dwarf": {"LSB","BCD","dG"},
    "Regular galaxies": {"G","E","S0","S","SA","SB","SAB","Im","IG","EmG"},
}
rev = {code: fam for fam, codes in family_bins.items() for code in codes}
gals_df["otype_family"] = gals_df["otype"].map(rev).fillna("Other")

family_counts = (gals_df["otype_family"]
                 .value_counts()
                 .rename_axis("otype_family")
                 .reset_index(name="count"))
family_counts


,otype_family,count
0,Regular galaxies,68
1,Members of systems,65
2,Other,29
3,Active (AGN/Seyfert/QSO),10
4,LSB/BCD/Dwarf,2


In [17]:
from astroquery.simbad import Simbad
s = Simbad(); 
s.add_votable_fields("morphtype")

morph = s.query_objects(gals_df["name"].tolist()).to_pandas()[["main_id","morph_type"]] 
morph = morph.rename(columns={"main_id": "name"})
morph
gals_df = gals_df.merge(morph, on="name", how="left")


In [18]:
is_E   = (gals_df["otype"] == "E")
is_BCG = (gals_df["otype"] == "BiC")  # SIMBAD code for brightest cluster galaxy
has_cD = gals_df["morph_type"].astype("string").str.contains(r"\bcD\b", na=False)


In [19]:
from astropy.cosmology import Planck15 as cosmo
import numpy as np

# distance modulus & angular diameter distance
z = gals_df["z"].astype(float)
DM = cosmo.distmod(z).value                 # mag
DM = 35.0  # assumed value 
DA = cosmo.angular_diameter_distance(z).to("kpc").value  # kpc

# absolute M_V (ignore extinction unless you have it)
# gals_df["M_V"] = gals_df["Vmag"].astype(float) - DM

# crude physical size from your major-axis measurement if present (arcsec → kpc)
maj_arcsec = pd.to_numeric(gals_df.get("major_axis_arcsec"), errors="coerce")
gals_df["major_kpc"] = (maj_arcsec/206265.0) * (DA*1e3)  # 1 rad = 206265"

is_luminous = gals_df["MV"] <= -21.5
is_huge     = gals_df["major_kpc"] >= 30                  # very extended on the sky


In [20]:
gals_df[is_luminous]

,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV,show_label,Re_arcsec,otype_family,morph_type,major_kpc
23,NGC 4889,195.033738,27.977025,EmG,0.02150,11.30,175.404602,1.816900,75,SIMBAD,B,2023ApJS..269....3M,<NA>,-23.70,True,21.925575,Regular galaxies,NaN,78815.629278
33,NGC 4874,194.898789,27.959248,LIN,0.02391,12.71,140.404205,2.260740,63,SIMBAD,B,2023ApJS..269....3M,<NA>,-22.29,True,17.550526,Active (AGN/Seyfert/QSO),NaN,69955.709614
100,IC 4051,195.226929,28.007639,EmG,0.01662,13.50,79.127998,0.915775,103,SIMBAD,B,2023ApJS..269....3M,<NA>,-21.50,True,9.891000,Regular galaxies,E3,27648.455901


In [21]:
gals_df[is_huge]

,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,galdim_qual,galdim_bibcode,dim_ref,MV,show_label,Re_arcsec,otype_family,morph_type,major_kpc
0,LEDA 3098454,195.075410,27.956592,GiC,0.021240,14.333000,23.714220,0.229277,143,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.667000,False,2.964278,Members of systems,SB0:,10530.120364
1,SDSS J125939.47+275116.5,194.914488,27.854598,G,0.031340,16.944000,1.300000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-18.056000,False,0.162500,Regular galaxies,,841.404907
2,LEDA 126763,195.055833,28.053417,GiC,0.027160,17.309999,23.923620,0.110128,81,SIMBAD,B,2023ApJS..269....3M,<NA>,-17.690001,False,2.990453,Members of systems,NaN,13486.880114
3,GMP 3424,194.872189,27.942225,GiC,0.018723,18.773001,4.070000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-16.226999,False,0.508750,Members of systems,dE,1597.967878
4,LEDA 126771,195.015438,27.964525,GiC,0.017780,17.260000,7.150000,NaN,<NA>,NED,,,2007SDSS6.C...0000:,-17.740000,False,0.893750,Members of systems,NaN,2668.911659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,IC 4021,195.061455,28.041300,LIN,0.019120,14.750000,25.321800,0.362693,49,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.250000,True,3.165225,Active (AGN/Seyfert/QSO),E3,10147.779410
169,LEDA 126761,195.077274,28.097154,G,0.026080,17.040001,15.108181,0.185831,172,SIMBAD,B,2023ApJS..269....3M,<NA>,-17.959999,False,1.888523,Regular galaxies,NaN,8189.209492
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.110000,False,5.332778,Members of systems,E1,23098.786248
171,IC 3973,194.878430,27.884206,AG?,0.015710,14.030000,49.606979,0.516492,137,SIMBAD,B,2023ApJS..269....3M,<NA>,-20.970000,True,6.200872,Other,SBa,16402.501923


In [22]:
gals_df["is_large_E"] = (is_E & (is_luminous | is_huge)) | is_BCG | has_cD


In [23]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord, match_coordinates_sky
import astropy.units as u
from astroquery.sdss import SDSS

def attach_sdss_re(gals_df, ra_col="ra", dec_col="dec", band="r", radius="3 arcsec", dr=17):
    df = gals_df.copy()
    coords = SkyCoord(df[ra_col].values, df[dec_col].values, unit='deg', frame='icrs')

    # Ask SDSS CrossID for the size fields we need (one nearest primary per input by default)
    fields = [
        f"deVRad_{band}", f"deVAB_{band}",  # de Vauc effective radius & b/a
        f"expRad_{band}", f"expAB_{band}",  # exponential effective radius & b/a (SDSS defines expRad as Re)
        f"fracDeV_{band}",                  # fraction of deV in the cModel (decision helper)
        "type", "mode", "objid", "ra", "dec"
    ]
    xid = SDSS.query_crossid(coords, radius=radius, photoobj_fields=fields, data_release=dr)
    if xid is None or len(xid) == 0:
        # nothing matched; just return original df with empty columns
        for c in ["Re_r_arcsec", "Re_r_arcsec_circ", "sdss_objid"]:
            df[c] = pd.NA
        return df

    t = xid.to_pandas()

    # Prefer primary galaxy detections if columns exist
    if "mode" in t.columns:
        t = t[t["mode"] == 1]
    if "type" in t.columns:
        t = t[t["type"] == 3]  # 3=GALAXY

    # Match returned SDSS rows back to input rows by nearest on-sky
    sdss_sc = SkyCoord(t["ra"].values, t["dec"].values, unit='deg')
    idx, sep, _ = match_coordinates_sky(coords, sdss_sc)
    # Trust matches within the requested radius
    max_sep = u.Quantity(radius)
    good = sep <= max_sep

    # Prepare result series aligned to input
    sel = t.iloc[idx].reset_index(drop=True)
    sel.loc[~good, :] = np.nan  # blank out non-matches

    # Choose an Re per object:
    # - SDSS defines deVRad_* as half-light (effective) radius for de Vaucouleurs fits
    # - SDSS defines expRad_* as half-light (effective) radius for exponential fits
    frac = sel.get(f"fracDeV_{band}")
    Re_deV = sel.get(f"deVRad_{band}")
    Re_exp = sel.get(f"expRad_{band}")

    use_deV = frac.notna() & (frac >= 0.5)
    Re = np.where(use_deV, Re_deV, Re_exp)

    # Circularize using the matching b/a
    ba = np.where(use_deV, sel.get(f"deVAB_{band}"), sel.get(f"expAB_{band}"))
    Re_circ = Re * np.sqrt(ba)

    # Attach to your DataFrame
    df["sdss_objid"]       = sel.get("objid").values
    df["Re_r_arcsec"]      = pd.to_numeric(Re, errors="coerce")
    df["Re_r_arcsec_circ"] = pd.to_numeric(Re_circ, errors="coerce")

    return df



In [24]:
gals_df = attach_sdss_re(gals_df, ra_col="ra", dec_col="dec", band="r", radius="3 arcsec", dr=17)


In [25]:
gals_df.sort_values(by='Re_r_arcsec_circ', ascending=False).head(30)


,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,...,MV,show_label,Re_arcsec,otype_family,morph_type,major_kpc,is_large_E,sdss_objid,Re_r_arcsec,Re_r_arcsec_circ
33,NGC 4874,194.898789,27.959248,LIN,0.023910,12.710000,140.404205,2.260740,63,SIMBAD,...,-22.290000,True,17.550526,Active (AGN/Seyfert/QSO),NaN,69955.709614,False,1.237667e+18,29.679890,29.586330
23,NGC 4889,195.033738,27.977025,EmG,0.021500,11.300000,175.404602,1.816900,75,SIMBAD,...,-23.700000,True,21.925575,Regular galaxies,NaN,78815.629278,False,1.237667e+18,29.678560,23.026233
60,LEDA 44708,195.025448,27.978315,GiC,0.025460,15.994000,33.000000,0.190000,141,SIMBAD,...,-19.006000,False,4.125000,Members of systems,NaN,17475.131768,False,1.237667e+18,29.675960,21.289626
16,SDSS J130051.15+280249.7,195.213151,28.047139,GiC,0.021160,17.077000,15.450000,NaN,<NA>,NED,...,-17.923000,False,1.931250,Members of systems,,6835.279943,False,1.237667e+18,16.102630,14.125548
100,IC 4051,195.226929,28.007639,EmG,0.016620,13.500000,79.127998,0.915775,103,SIMBAD,...,-21.500000,True,9.891000,Regular galaxies,E3,27648.455901,False,1.237667e+18,16.322610,14.110691
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72,SIMBAD,...,-20.110000,False,5.332778,Members of systems,E1,23098.786248,False,1.237667e+18,14.073110,13.508782
142,SDSS J130042.56+280658.6,195.177360,28.116320,LSB,0.020701,18.240000,19.498001,0.324967,0,SIMBAD,...,-16.760000,False,2.437250,LSB/BCD/Dwarf,,8443.755209,False,1.237667e+18,11.911510,10.328826
50,LEDA 44636,194.909672,27.987209,G,0.022740,15.890000,16.380001,0.191000,40,SIMBAD,...,-19.110000,False,2.047500,Regular galaxies,NaN,7772.914363,False,1.237667e+18,11.870460,9.669536
136,LEDA 126768,195.031143,27.958069,GiC,0.020720,17.580000,13.800000,0.210000,<NA>,SIMBAD,...,-17.420000,False,1.725000,Members of systems,NaN,5981.540402,False,1.237667e+18,9.742112,8.978572
122,LEDA 126762,195.056811,27.867177,GiC,0.024680,16.350000,18.000000,0.210000,109,SIMBAD,...,-18.650000,False,2.250000,Members of systems,NaN,9248.599687,False,1.237667e+18,9.405168,8.298744


In [26]:
def attach_gc_Re_defaults(gals_df):
    df = gals_df.copy()
    # pick galaxy Re (prefer circularized)
    Re_gal_arcsec = df.get("Re_r_arcsec_circ").fillna(df.get("Re_r_arcsec"))
    # scaling factor by class
    scale = np.where(df.get("is_large_E", False), 4.8, np.where(df["otype"].eq("E"), 3.2, 3.0))
    df["Re_GCS_arcsec"] = Re_gal_arcsec.astype(float) * scale

    return df

gals_df = attach_gc_Re_defaults(gals_df)


In [27]:
gals_df.sort_values(by='Re_r_arcsec_circ', ascending=False).head(30)


,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,...,show_label,Re_arcsec,otype_family,morph_type,major_kpc,is_large_E,sdss_objid,Re_r_arcsec,Re_r_arcsec_circ,Re_GCS_arcsec
33,NGC 4874,194.898789,27.959248,LIN,0.023910,12.710000,140.404205,2.260740,63,SIMBAD,...,True,17.550526,Active (AGN/Seyfert/QSO),NaN,69955.709614,False,1.237667e+18,29.679890,29.586330,88.758989
23,NGC 4889,195.033738,27.977025,EmG,0.021500,11.300000,175.404602,1.816900,75,SIMBAD,...,True,21.925575,Regular galaxies,NaN,78815.629278,False,1.237667e+18,29.678560,23.026233,69.078698
60,LEDA 44708,195.025448,27.978315,GiC,0.025460,15.994000,33.000000,0.190000,141,SIMBAD,...,False,4.125000,Members of systems,NaN,17475.131768,False,1.237667e+18,29.675960,21.289626,63.868877
16,SDSS J130051.15+280249.7,195.213151,28.047139,GiC,0.021160,17.077000,15.450000,NaN,<NA>,NED,...,False,1.931250,Members of systems,,6835.279943,False,1.237667e+18,16.102630,14.125548,42.376644
100,IC 4051,195.226929,28.007639,EmG,0.016620,13.500000,79.127998,0.915775,103,SIMBAD,...,True,9.891000,Regular galaxies,E3,27648.455901,False,1.237667e+18,16.322610,14.110691,42.332073
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72,SIMBAD,...,False,5.332778,Members of systems,E1,23098.786248,False,1.237667e+18,14.073110,13.508782,40.526346
142,SDSS J130042.56+280658.6,195.177360,28.116320,LSB,0.020701,18.240000,19.498001,0.324967,0,SIMBAD,...,False,2.437250,LSB/BCD/Dwarf,,8443.755209,False,1.237667e+18,11.911510,10.328826,30.986478
50,LEDA 44636,194.909672,27.987209,G,0.022740,15.890000,16.380001,0.191000,40,SIMBAD,...,False,2.047500,Regular galaxies,NaN,7772.914363,False,1.237667e+18,11.870460,9.669536,29.008607
136,LEDA 126768,195.031143,27.958069,GiC,0.020720,17.580000,13.800000,0.210000,<NA>,SIMBAD,...,False,1.725000,Members of systems,NaN,5981.540402,False,1.237667e+18,9.742112,8.978572,26.935717
122,LEDA 126762,195.056811,27.867177,GiC,0.024680,16.350000,18.000000,0.210000,109,SIMBAD,...,False,2.250000,Members of systems,NaN,9248.599687,False,1.237667e+18,9.405168,8.298744,24.896231


In [28]:
import numpy as np

def sersic_n_priors(df, re_gcs_col="Re_GCS_arcsec",
                    re_gal_cols=("Re_r_arcsec_circ","Re_r_arcsec"),
                    is_bcg_col="is_bcg", mstar_col="Mstar"):
    out = df.copy()
    Re_gcs = out[re_gcs_col].astype(float)
    # pick a galaxy Re
    Re_gal = None
    for c in re_gal_cols:
        if c in out.columns:
            Re_gal = out[c]
            if Re_gal.notna().any(): break
    Re_gal = pd.to_numeric(Re_gal, errors="coerce")
    eta = Re_gcs / Re_gal

    # base n_total from eta (clip to safe range)
    n_total = 2.1 - 0.2*np.log(eta.replace([np.inf, -np.inf], np.nan))
    n_total = np.clip(n_total, 0.7, 3.0)

    # BCG/cD tweak: shallower by 0.2
    is_bcg = np.zeros(len(out), dtype=bool)
    if is_bcg_col in out.columns:
        is_bcg |= out[is_bcg_col].fillna(False).astype(bool).values
    if mstar_col in out.columns:
        is_bcg |= (pd.to_numeric(out[mstar_col], errors="coerce") >= 3e11).fillna(False).values
    n_total = np.where(is_bcg, n_total - 0.2, n_total)

    out["n_gcs_prior"]       = n_total
    out["n_gcs_red_prior"]   = np.clip(1.2*n_total, 0.8, 3.5)
    out["n_gcs_blue_prior"]  = np.clip(0.8*n_total, 0.6, 2.8)

    return out


In [29]:
gals_df = sersic_n_priors(gals_df)


In [30]:
gals_df.sort_values(by='Re_r_arcsec_circ', ascending=False).head(30)


,name,ra,dec,otype,z,mv,major_axis_arcsec,galdim_minaxis,galdim_angle,dim_source,...,morph_type,major_kpc,is_large_E,sdss_objid,Re_r_arcsec,Re_r_arcsec_circ,Re_GCS_arcsec,n_gcs_prior,n_gcs_red_prior,n_gcs_blue_prior
33,NGC 4874,194.898789,27.959248,LIN,0.023910,12.710000,140.404205,2.260740,63,SIMBAD,...,NaN,69955.709614,False,1.237667e+18,29.679890,29.586330,88.758989,1.880278,2.256333,1.504222
23,NGC 4889,195.033738,27.977025,EmG,0.021500,11.300000,175.404602,1.816900,75,SIMBAD,...,NaN,78815.629278,False,1.237667e+18,29.678560,23.026233,69.078698,1.880278,2.256333,1.504222
60,LEDA 44708,195.025448,27.978315,GiC,0.025460,15.994000,33.000000,0.190000,141,SIMBAD,...,NaN,17475.131768,False,1.237667e+18,29.675960,21.289626,63.868877,1.880278,2.256333,1.504222
16,SDSS J130051.15+280249.7,195.213151,28.047139,GiC,0.021160,17.077000,15.450000,NaN,<NA>,NED,...,,6835.279943,False,1.237667e+18,16.102630,14.125548,42.376644,1.880278,2.256333,1.504222
100,IC 4051,195.226929,28.007639,EmG,0.016620,13.500000,79.127998,0.915775,103,SIMBAD,...,E3,27648.455901,False,1.237667e+18,16.322610,14.110691,42.332073,1.880278,2.256333,1.504222
170,2MASX J12591389+2804349,194.808025,28.076248,GiC,0.026050,14.890000,42.662220,0.687217,72,SIMBAD,...,E1,23098.786248,False,1.237667e+18,14.073110,13.508782,40.526346,1.880278,2.256333,1.504222
142,SDSS J130042.56+280658.6,195.177360,28.116320,LSB,0.020701,18.240000,19.498001,0.324967,0,SIMBAD,...,,8443.755209,False,1.237667e+18,11.911510,10.328826,30.986478,1.880278,2.256333,1.504222
50,LEDA 44636,194.909672,27.987209,G,0.022740,15.890000,16.380001,0.191000,40,SIMBAD,...,NaN,7772.914363,False,1.237667e+18,11.870460,9.669536,29.008607,1.880278,2.256333,1.504222
136,LEDA 126768,195.031143,27.958069,GiC,0.020720,17.580000,13.800000,0.210000,<NA>,SIMBAD,...,NaN,5981.540402,False,1.237667e+18,9.742112,8.978572,26.935717,1.880278,2.256333,1.504222
122,LEDA 126762,195.056811,27.867177,GiC,0.024680,16.350000,18.000000,0.210000,109,SIMBAD,...,NaN,9248.599687,False,1.237667e+18,9.405168,8.298744,24.896231,1.880278,2.256333,1.504222


In [31]:
gals_df.to_csv('data/gals_data_from_archives.csv', index=False)


In [32]:
gals_df.loc[gals_df['name'] == 'Coma cluster'].iloc[0]


name                 Coma cluster
ra                     195.017071
dec                     27.977025
otype                     CLUSTER
z                          0.0215
mv                            NaN
major_axis_arcsec             NaN
galdim_minaxis                NaN
galdim_angle                 <NA>
dim_source                 MANUAL
galdim_qual                   NaN
galdim_bibcode                NaN
dim_ref                       NaN
MV                            NaN
show_label                  False
Re_arcsec                     NaN
otype_family                Other
morph_type                    NaN
major_kpc                     NaN
is_large_E                  False
sdss_objid                    NaN
Re_r_arcsec                   NaN
Re_r_arcsec_circ              NaN
Re_GCS_arcsec                 NaN
n_gcs_prior                   NaN
n_gcs_red_prior               NaN
n_gcs_blue_prior              NaN
Name: 173, dtype: object